# 08 - Calibracion e incertidumbre de clasificadores

Fase adicional ligera del TFM: analizar si las probabilidades producidas por los clasificadores reflejan la fiabilidad real de sus predicciones.

Este notebook no reentrena modelos. Lee los CSV de predicciones ya generados en la fase de clasificacion y calcula metricas de calibracion como ECE, Brier score, confianza media y errores de alta confianza.

In [ ]:
from pathlib import Path
import subprocess
import sys

import pandas as pd
from IPython.display import Image, Markdown, display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

CALIBRATION_DIR = PROJECT_ROOT / 'results' / 'calibration'
FIGURES_DIR = CALIBRATION_DIR / 'figures'
PYTHON_BIN = PROJECT_ROOT / '.conda' / 'bin' / 'python'
PYTHON_BIN = str(PYTHON_BIN if PYTHON_BIN.exists() else sys.executable)

PROJECT_ROOT

## Regenerar analisis de calibracion

El script `scripts/build_calibration_analysis.py` genera:

- `calibration_metrics.csv`: ECE, Brier score, NLL, confianza media y errores de alta confianza por experimento;
- `calibration_bins.csv`: accuracy y confianza media por bin;
- `high_confidence_errors_top100.csv`: errores ordenados por confianza;
- figuras de reliability diagrams, histogramas de confianza y ECE por experimento;
- `calibration_summary.md`: lectura metodologica resumida.

In [ ]:
RUN_BUILD = True

if RUN_BUILD:
    subprocess.run(
        [PYTHON_BIN, str(PROJECT_ROOT / 'scripts' / 'build_calibration_analysis.py')],
        cwd=PROJECT_ROOT,
        check=True,
    )

## Cargar resultados

In [ ]:
metrics_df = pd.read_csv(CALIBRATION_DIR / 'calibration_metrics.csv')
bins_df = pd.read_csv(CALIBRATION_DIR / 'calibration_bins.csv')
selected_df = pd.read_csv(CALIBRATION_DIR / 'selected_models.csv')
errors_df = pd.read_csv(CALIBRATION_DIR / 'high_confidence_errors_top100.csv')

display(Markdown('### Modelos representativos'))
display(selected_df[['dataset', 'experiment', 'architecture', 'balance_strategy', 'accuracy', 'f1_macro']])

display(Markdown('### Metricas de calibracion por experimento'))
display(
    metrics_df[
        [
            'dataset',
            'experiment',
            'accuracy',
            'f1_macro',
            'mean_confidence',
            'ece',
            'brier_score',
            'negative_log_likelihood',
            'high_confidence_error_count',
            'high_confidence_count',
        ]
    ].sort_values(['dataset', 'ece'])
)

## Figuras

La diagonal del reliability diagram representa calibracion ideal: si un grupo de predicciones tiene confianza media 0.8, deberia acertar aproximadamente el 80% de los casos.

In [ ]:
for figure_name in [
    'reliability_diagrams_selected_models.png',
    'confidence_histograms_selected_models.png',
    'ece_by_experiment.png',
]:
    display(Markdown(f'### {figure_name}'))
    display(Image(filename=str(FIGURES_DIR / figure_name)))

## Errores de alta confianza

Estos casos son relevantes en imagen medica porque representan predicciones incorrectas que el modelo presenta como seguras.

In [ ]:
display(errors_df.head(20))

## Resumen interpretativo

In [ ]:
display(Markdown((CALIBRATION_DIR / 'calibration_summary.md').read_text()))

## Lectura para la memoria

La calibracion complementa las metricas clasicas de clasificacion. Un modelo puede tener buena accuracy y, aun asi, estar mal calibrado si sus probabilidades no reflejan la frecuencia real de acierto. En un contexto medico, esto es especialmente relevante porque una prediccion erronea con alta confianza puede ser mas problematica que una prediccion erronea con baja confianza.

En este TFM, este analisis se puede presentar como una extension metodologica ligera: se aprovechan las predicciones ya guardadas para estudiar la fiabilidad probabilistica de los clasificadores CNN, sin reentrenar modelos. La comparacion entre CXR y CT permite discutir si la modalidad mas dificil tambien produce incertidumbre o sobreconfianza distinta.